In [3]:
"""
compare_qcnn_qiskit_transpiler_print.py

Confronto risorse dei circuiti QCNN usando il transpiler Qiskit/IBM.

Confronta le architetture registrate in suitev2:
    - hur6
    - hur8
    - hur9
    - custom_ansatz

per i tre encoding registrati in suitev2:
    - e3 amplitude encoding
    - e1 affine angle encoding + global ancilla
    - custom learned multiaxis pairwise fragment encoding

Per ogni combinazione:
    1. costruisce il circuito Qiskit
    2. lo transpila con generate_preset_pass_manager(...)
    3. stampa circuito raw, circuito transpiled, count_ops, depth, size, ecc.

Non salva CSV.

NOTE:
- Il codice usa solo backend fake IBM locali, senza accesso ad account IBM.
- Non esiste fallback locale manuale: se i fake backend non sono disponibili,
  il programma stampa un errore e termina.
"""

import random
from collections import OrderedDict

import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


# ============================================================
# Config
# ============================================================

SEED = 42

N_QUBITS = 8
E1_N_QUBITS = 9
IMG_SIZE = 16
N_CLASSES = 4

# Livello di ottimizzazione Qiskit: 0, 1, 2, 3
OPTIMIZATION_LEVEL = 3

# Fake backend IBM locali da provare.
# Tieni solo quelli che vuoi usare nel confronto.
FAKE_BACKEND_NAMES = [
    "FakeSherbrooke",
    "FakeTorino",
]

# Hur architectures: no final classifier; hur9 also skips pooling, matching suitev2.
USE_FINAL_CLASSIFIER_FOR_HUR = False

# Custom architecture: transfer pooling + final two-qubit classifier.
# Always enabled for the custom architecture.

# Custom encoding config
PATCH_SIZE = 8
SUBPATCH_SIZE = 2
N_PATCH_GROUPS = 2
FEATURES_PER_PATCH = 16
FEATURES_PER_ENCODING_STEP = 4
N_ENCODING_STEPS = FEATURES_PER_PATCH // FEATURES_PER_ENCODING_STEP

# E1 encoding config, aligned with suitev2/config.py
E1_INIT_A = 0.2
LAMBDA_FUSION = np.pi / 4
E1_OMEGA_FIXED = np.pi / 2
BETA_GLOBAL = np.array([1.0, 10.0, 10.0, 1.0], dtype=float)


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)


# ============================================================
# Dummy inputs and parameters
# ============================================================

def random_angles(rng, shape, low=-np.pi, high=np.pi):
    """Deterministic random dummy trainable parameters for transpilation."""
    return rng.uniform(low, high, size=shape).astype(float)


def make_dummy_amplitude_input():
    """
    One 16x16 image flattened to 256 amplitudes.
    Normalized because QuantumCircuit.initialize expects a statevector.
    """
    rng = np.random.default_rng(SEED)
    x = rng.random(IMG_SIZE * IMG_SIZE)
    x = x / np.linalg.norm(x)
    return x


def patch_8x8_to_2x2_means_single(patch):
    """
    One 8x8 patch -> sixteen local 2x2 means.

    Input:
        patch shape: (8, 8)

    Output:
        features shape: (16,)
    """
    patch = patch.reshape(4, 2, 4, 2)
    means = patch.mean(axis=(1, 3))
    return means.reshape(-1)


def image_to_four_patch_features_single(img):
    """
    One 16x16 image -> four patch feature vectors.

    Output:
        patches shape: (4, 16)

    Patch order:
        0 = top-left
        1 = top-right
        2 = bottom-left
        3 = bottom-right
    """
    p0 = img[0:8, 0:8]
    p1 = img[0:8, 8:16]
    p2 = img[8:16, 0:8]
    p3 = img[8:16, 8:16]

    f0 = patch_8x8_to_2x2_means_single(p0)
    f1 = patch_8x8_to_2x2_means_single(p1)
    f2 = patch_8x8_to_2x2_means_single(p2)
    f3 = patch_8x8_to_2x2_means_single(p3)

    patches = np.stack([f0, f1, f2, f3], axis=0)

    # Feature values in [0,1] -> angles in [0, pi].
    return np.pi * patches


def make_dummy_fragment_input():
    rng = np.random.default_rng(SEED)
    img = rng.random((IMG_SIZE, IMG_SIZE))
    return image_to_four_patch_features_single(img)


def extract_quad_means_8_single(img):
    """One 16x16 image -> 8 local means on a 2x4 grid."""
    blocks = []
    for row in range(2):
        for col in range(4):
            block = img[row * 8:(row + 1) * 8, col * 4:(col + 1) * 4]
            blocks.append(block.mean())
    return np.array(blocks, dtype=float)


def extract_gA4_single(img):
    """One 16x16 image -> suitev2 E1 global statistics."""
    mean = img.mean()
    var = img.var()

    gx = img[:, 1:] - img[:, :-1]
    gy = img[1:, :] - img[:-1, :]
    grad_energy = (np.mean(gx ** 2) + np.mean(gy ** 2)) / 2

    h_var = img.var(axis=1).mean()
    v_var = img.var(axis=0).mean()
    hv_asym = (h_var - v_var) / (h_var + v_var + 1e-8)

    return np.array([mean - 0.5, var, grad_energy, hv_asym], dtype=float)


def make_dummy_e1_input():
    rng = np.random.default_rng(SEED)
    img = rng.random((IMG_SIZE, IMG_SIZE))
    return extract_quad_means_8_single(img), extract_gA4_single(img)


def compute_gamma(gA4):
    return np.pi * np.tanh(BETA_GLOBAL * gA4)


# ============================================================
# Small Qiskit helpers
# ============================================================

def u3_compat(qc: QuantumCircuit, theta, phi, lam, qubit):
    """
    Qiskit moved from u3 to u. This helper works across versions.
    """
    if hasattr(qc, "u"):
        qc.u(theta, phi, lam, qubit)
    else:
        qc.u3(theta, phi, lam, qubit)


def append_measurements(qc: QuantumCircuit, output_qubits):
    """
    Add classical bits and measure only output qubits.
    """
    creg = ClassicalRegister(len(output_qubits), "c")
    qc.add_register(creg)
    for bit_idx, wire in enumerate(output_qubits):
        qc.measure(wire, creg[bit_idx])
    return qc


def two_qubit_gate_count(qc: QuantumCircuit):
    """
    Count operations acting on exactly 2 qubits.
    """
    total = 0
    for inst in qc.data:
        op = inst.operation
        if getattr(op, "num_qubits", 0) == 2:
            total += 1
    return total


def depth_2q(qc: QuantumCircuit):
    """
    Qiskit depth restricted to 2-qubit operations.
    Compatible with recent Qiskit versions.
    """
    try:
        return qc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)
    except TypeError:
        # Older Qiskit may pass tuple-like CircuitInstruction or old tuple.
        return qc.depth(filter_function=lambda inst: inst[0].num_qubits == 2)


def print_resource_report(label, qc: QuantumCircuit):
    """
    Print Qiskit resource summary.
    """
    print(f"\n[{label}]")
    print("num_qubits:", qc.num_qubits)
    print("num_clbits:", qc.num_clbits)
    print("size:", qc.size())
    print("depth:", qc.depth())
    print("depth_2q:", depth_2q(qc))
    print("2q_gate_count:", two_qubit_gate_count(qc))
    print("count_ops:", dict(qc.count_ops()))


# ============================================================
# Hur convolutional ansatz
# ============================================================

def hur_convolution_circuit6(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 6 = U_SO4.
    6 parameters.
    """
    a, b = wires
    qc.ry(theta[0], a)
    qc.ry(theta[1], b)
    qc.cx(a, b)
    qc.ry(theta[2], a)
    qc.ry(theta[3], b)
    qc.cx(a, b)
    qc.ry(theta[4], a)
    qc.ry(theta[5], b)


def hur_convolution_circuit8(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 8.
    10 parameters.
    """
    a, b = wires
    qc.rx(theta[0], a)
    qc.rx(theta[1], b)
    qc.rz(theta[2], a)
    qc.rz(theta[3], b)
    qc.rx(theta[4], a)
    qc.rx(theta[5], b)
    qc.cx(a, b)
    qc.rx(theta[6], a)
    qc.rx(theta[7], b)
    qc.rz(theta[8], a)
    qc.rz(theta[9], b)


def hur_convolution_circuit9(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 9 = U_SU4.
    15 parameters.
    """
    a, b = wires
    u3_compat(qc, theta[0], theta[1], theta[2], a)
    u3_compat(qc, theta[3], theta[4], theta[5], b)
    qc.cx(a, b)
    qc.ry(theta[6], a)
    qc.rz(theta[7], b)
    qc.cx(b, a)
    qc.ry(theta[8], a)
    qc.cx(a, b)
    u3_compat(qc, theta[9], theta[10], theta[11], a)
    u3_compat(qc, theta[12], theta[13], theta[14], b)


# ============================================================
# Custom Cartan convolution + transfer pooling
# ============================================================

def custom_cartan_convolution_block(qc: QuantumCircuit, theta, wires):
    """
    Custom Cartan-inspired convolution:
        local rotations -> RXX/RYY/RZZ -> local rotations

    11 parameters.
    """
    a, b = wires

    qc.rz(theta[0], a)
    qc.ry(theta[1], a)
    qc.rz(theta[2], b)
    qc.ry(theta[3], b)

    qc.rxx(theta[4], a, b)
    qc.ryy(theta[5], a, b)
    qc.rzz(theta[6], a, b)

    qc.ry(theta[7], a)
    qc.rz(theta[8], a)
    qc.ry(theta[9], b)
    qc.rz(theta[10], b)


def transfer_pooling_pair(qc: QuantumCircuit, theta_pool, discard, keep):
    """
    Custom controlled-transfer pooling.
    3 parameters.
    """
    qc.cx(discard, keep)
    qc.cry(theta_pool[0], discard, keep)
    qc.crz(theta_pool[1], discard, keep)
    qc.ry(theta_pool[2], keep)


def transfer_pooling_layer(qc: QuantumCircuit, theta_pool, pool_pairs):
    for discard, keep in pool_pairs:
        transfer_pooling_pair(qc, theta_pool, discard, keep)


# ============================================================
# Hur pooling
# ============================================================

def controlled_rx_on_zero(qc: QuantumCircuit, angle, control, target):
    """
    Controlled-RX activated when control is |0>.
    """
    qc.x(control)
    qc.crx(angle, control, target)
    qc.x(control)


def hur_pool_pair(qc: QuantumCircuit, theta_pool, control, target):
    """
    Hur pooling:
        CRZ(theta[0]) with control |1>
        CRX(theta[1]) with control |0>
    """
    qc.crz(theta_pool[0], control, target)
    controlled_rx_on_zero(qc, theta_pool[1], control, target)


def hur_pool_layer(qc: QuantumCircuit, theta_pool, pool_pairs):
    for control, target in pool_pairs:
        hur_pool_pair(qc, theta_pool, control, target)


# ============================================================
# Common convolution layer driver
# ============================================================

def convolution_layer_on_wires(qc: QuantumCircuit, conv_fn, theta_conv, active_wires):
    """
    Apply same two-qubit convolution block on even and shifted pairs.

    No periodic boundary is used:
        [0,1,2,3,4,5,6,7]
        -> even pairs:    (0,1), (2,3), (4,5), (6,7)
        -> shifted pairs: (1,2), (3,4), (5,6)
    """
    even_pairs = [
        (active_wires[i], active_wires[i + 1])
        for i in range(0, len(active_wires) - 1, 2)
    ]

    shifted_pairs = [
        (active_wires[i], active_wires[i + 1])
        for i in range(1, len(active_wires) - 1, 2)
    ]

    for pair in even_pairs:
        conv_fn(qc, theta_conv, pair)

    for pair in shifted_pairs:
        conv_fn(qc, theta_conv, pair)


# ============================================================
# Encodings
# ============================================================

def amplitude_encoding(qc: QuantumCircuit, x):
    """
    Classic amplitude encoding via Qiskit initialize.

    No extra norm/brightness rotations are applied after initialize. The
    initialize instruction is kept intact here so the Qiskit transpiler shows
    the real synthesized cost after decomposing it for the chosen backend.
    """
    qc.initialize(x, list(range(N_QUBITS)))


def e1_encoding(qc: QuantumCircuit, e1_features, a_embed, c_embed,
                ancilla_wire=8, lam=LAMBDA_FUSION, omega_fixed=E1_OMEGA_FIXED):
    """suitev2 E1: affine local RY + global ancilla re-upload + fusion."""
    quad_means_8, gA4 = e1_features
    gammas_4 = compute_gamma(gA4)

    for i in range(8):
        angle = a_embed[i] * (np.pi * quad_means_8[i]) + c_embed[i]
        qc.ry(angle, i)

    qc.ry(gammas_4[0], ancilla_wire)
    qc.rz(gammas_4[1], ancilla_wire)
    qc.rx(gammas_4[2], ancilla_wire)
    qc.rz(gammas_4[3], ancilla_wire)
    qc.ry(omega_fixed, ancilla_wire)
    qc.rz(gammas_4[0], ancilla_wire)
    qc.rx(gammas_4[1], ancilla_wire)
    qc.ry(gammas_4[2], ancilla_wire)
    qc.rz(gammas_4[3], ancilla_wire)

    for i in range(8):
        qc.cx(ancilla_wire, i)
        qc.rz(lam, i)
        qc.cx(ancilla_wire, i)


def encode_patch_on_qubit_pair_multiaxis(qc: QuantumCircuit, patch, theta_enc_group, wires):
    """New proposed encoding from nuovo_encoding_pazzo.ipynb, translated to Qiskit."""
    a, b = wires

    for s in range(N_ENCODING_STEPS):
        base = 4 * s
        x0 = patch[base + 0]
        x1 = patch[base + 1]
        x2 = patch[base + 2]
        x3 = patch[base + 3]

        z0 = theta_enc_group[s, 0] * x0 + theta_enc_group[s, 1]
        z1 = theta_enc_group[s, 2] * x1 + theta_enc_group[s, 3]
        z2 = theta_enc_group[s, 4] * x2 + theta_enc_group[s, 5]
        z3 = theta_enc_group[s, 6] * x3 + theta_enc_group[s, 7]

        qc.ry(z0, a)
        qc.rz(z1, a)
        qc.rx(z2, b)
        qc.ry(z3, b)

        qc.cx(a, b)
        qc.ry(theta_enc_group[s, 8], a)
        qc.rz(theta_enc_group[s, 9], b)

        qc.cx(b, a)
        qc.rx(theta_enc_group[s, 10], a)
        qc.ry(theta_enc_group[s, 11], b)


def learned_multiaxis_fragment_encoding(qc: QuantumCircuit, patches, theta_enc):
    """Four-patch top/bottom-shared multiaxis encoding from nuovo_encoding_pazzo.ipynb."""
    pair_wires = [
        (0, 1),
        (2, 3),
        (4, 5),
        (6, 7),
    ]

    for patch_idx, wires in enumerate(pair_wires):
        group_idx = 0 if patch_idx in [0, 1] else 1
        encode_patch_on_qubit_pair_multiaxis(
            qc,
            patches[patch_idx],
            theta_enc[group_idx],
            wires,
        )


def apply_encoding(qc: QuantumCircuit, encoding_name, x, theta_enc,
                   a_embed, c_embed):
    if encoding_name == "e3":
        amplitude_encoding(qc, x)
    elif encoding_name == "e1":
        e1_encoding(qc, x, a_embed, c_embed)
    elif encoding_name == "custom":
        learned_multiaxis_fragment_encoding(qc, x, theta_enc)
    else:
        raise ValueError(f"Unknown encoding: {encoding_name}")


# ============================================================
# QCNN architecture builders
# ============================================================

def apply_qcnn_hur(qc: QuantumCircuit, conv_fn, theta_conv1, theta_pool1,
                   theta_conv2, theta_pool2, theta_final=None, use_param_pool=True):
    """
    Hur-style QCNN:
        conv -> optional Hur pooling 8->4
        conv -> optional Hur pooling 4->2
        optional final conv on output wires
    """
    wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]
    wires_4 = [0, 2, 4, 6]
    output_wires = [0, 4]

    convolution_layer_on_wires(qc, conv_fn, theta_conv1, wires_8)

    if use_param_pool:
        hur_pool_layer(
            qc,
            theta_pool1,
            pool_pairs=[
                (1, 0),
                (3, 2),
                (5, 4),
                (7, 6),
            ],
        )

    convolution_layer_on_wires(qc, conv_fn, theta_conv2, wires_4)

    if use_param_pool:
        hur_pool_layer(
            qc,
            theta_pool2,
            pool_pairs=[
                (2, 0),
                (6, 4),
            ],
        )

    if theta_final is not None:
        conv_fn(qc, theta_final, output_wires)

    return output_wires


def apply_qcnn_custom(qc: QuantumCircuit, theta_conv1, theta_pool1,
                      theta_conv2, theta_pool2, theta_final=None):
    """
    Custom QCNN:
        Cartan conv -> transfer pooling 8->4
        Cartan conv -> transfer pooling 4->2
        optional final Cartan classifier
    """
    wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]
    wires_4 = [0, 2, 4, 6]
    output_wires = [0, 4]

    convolution_layer_on_wires(
        qc,
        custom_cartan_convolution_block,
        theta_conv1,
        wires_8,
    )

    transfer_pooling_layer(
        qc,
        theta_pool1,
        pool_pairs=[
            (1, 0),
            (3, 2),
            (5, 4),
            (7, 6),
        ],
    )

    convolution_layer_on_wires(
        qc,
        custom_cartan_convolution_block,
        theta_conv2,
        wires_4,
    )

    transfer_pooling_layer(
        qc,
        theta_pool2,
        pool_pairs=[
            (2, 0),
            (6, 4),
        ],
    )

    if theta_final is not None:
        custom_cartan_convolution_block(qc, theta_final, output_wires)

    return output_wires


def build_qcnn_circuit(ansatz_name, ansatz_cfg, encoding_name):
    """
    Build raw Qiskit QuantumCircuit for one ansatz/encoding combination.
    """
    n_qubits = E1_N_QUBITS if encoding_name == "e1" else N_QUBITS
    qc = QuantumCircuit(n_qubits)

    if encoding_name == "e3":
        x = make_dummy_amplitude_input()
    elif encoding_name == "custom":
        x = make_dummy_fragment_input()
    elif encoding_name == "e1":
        x = make_dummy_e1_input()
    else:
        raise ValueError(f"Unknown encoding: {encoding_name}")

    param_key = f"{ansatz_name}:{encoding_name}"
    param_seed = SEED + 10_000 + sum((i + 1) * ord(ch) for i, ch in enumerate(param_key))
    param_rng = np.random.default_rng(param_seed)

    theta_enc = random_angles(param_rng, (N_PATCH_GROUPS, N_ENCODING_STEPS, 12))
    a_embed = random_angles(param_rng, 8)
    c_embed = random_angles(param_rng, 8)

    conv_params = ansatz_cfg["conv_params"]
    pool_params = ansatz_cfg["pool_params"]

    theta_conv1 = random_angles(param_rng, conv_params)
    theta_pool1 = random_angles(param_rng, pool_params)
    theta_conv2 = random_angles(param_rng, conv_params)
    theta_pool2 = random_angles(param_rng, pool_params)
    theta_final = random_angles(param_rng, conv_params)

    apply_encoding(qc, encoding_name, x, theta_enc, a_embed, c_embed)

    if ansatz_name == "custom_ansatz":
        output_wires = apply_qcnn_custom(
            qc,
            theta_conv1,
            theta_pool1,
            theta_conv2,
            theta_pool2,
            theta_final,
        )
    else:
        output_wires = apply_qcnn_hur(
            qc,
            ansatz_cfg["conv_fn"],
            theta_conv1,
            theta_pool1,
            theta_conv2,
            theta_pool2,
            theta_final if USE_FINAL_CLASSIFIER_FOR_HUR else None,
            use_param_pool=ansatz_name != "hur9",
        )

    # Match suitev2 readout: qml.probs(wires=[0, 4]).
    append_measurements(qc, output_wires)

    return qc, output_wires


# ============================================================
# Backend / compiler setup
# ============================================================

def load_fake_backend_class(class_name):
    """
    Load a fake IBM backend class by name from the available Qiskit providers.
    No real IBM account is used.
    """
    provider_modules = [
        "qiskit_ibm_runtime.fake_provider",
        "qiskit.providers.fake_provider",
    ]

    last_error = None

    for module_name in provider_modules:
        try:
            module = __import__(module_name, fromlist=[class_name])
            cls = getattr(module, class_name)
            return cls()
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        f"Fake backend {class_name} is not available in this Qiskit installation. "
        f"Last error: {type(last_error).__name__}: {last_error}"
    )


def get_fake_backends():
    """
    Load all fake backends listed in FAKE_BACKEND_NAMES.
    If none can be loaded, raise an error instead of using a manual fallback.
    """
    backends = []

    for name in FAKE_BACKEND_NAMES:
        try:
            backend = load_fake_backend_class(name)
            backends.append(backend)
        except Exception as exc:
            print(f"[ERROR] Unable to load {name}: {type(exc).__name__}: {exc}")

    if not backends:
        raise RuntimeError(
            "No fake IBM backend could be loaded. "
            "Install/update qiskit-ibm-runtime or edit FAKE_BACKEND_NAMES."
        )

    return backends


def transpile_with_qiskit_compiler(qc: QuantumCircuit, backend):
    """
    Use Qiskit's preset pass manager for a fake IBM backend.
    No manual fallback architecture is used.
    """
    if backend is None:
        raise RuntimeError("backend is None: no fake IBM backend was loaded.")

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=OPTIMIZATION_LEVEL,
    )
    return pm.run(qc)


def print_backend_info(backend):
    """
    Print useful backend info.
    """
    print("" + "=" * 120)
    print("BACKEND INFO")
    print("=" * 120)

    if backend is None:
        print("[ERROR] Backend is None. No fake backend is available.")
        return

    print("backend:", backend)

    try:
        print("backend name:", backend.name)
    except Exception:
        try:
            print("backend name:", backend.name())
        except Exception:
            pass

    try:
        print("basis_gates:", backend.configuration().basis_gates)
    except Exception as exc:
        print("basis_gates: unavailable via backend.configuration()", exc)

    try:
        print("operation_names / supported instructions:", backend.operation_names)
    except Exception:
        pass

    try:
        print("num_qubits:", backend.num_qubits)
    except Exception:
        try:
            print("num_qubits:", backend.configuration().num_qubits)
        except Exception:
            pass

    try:
        print("coupling_map:", backend.coupling_map)
    except Exception:
        try:
            print("coupling_map:", backend.configuration().coupling_map)
        except Exception:
            pass


# ============================================================
# Run comparison
# ============================================================

def run_one_case(ansatz_name, ansatz_cfg, encoding_name, backend):
    title = f"{ansatz_name} / {encoding_name}"

    print("\n" + "=" * 120)
    print("CASE:", title)
    print("=" * 120)

    qc_raw, output_wires = build_qcnn_circuit(ansatz_name, ansatz_cfg, encoding_name)

    print("Output wires used for class probabilities before final measurement:", output_wires)

    print_resource_report("RAW CIRCUIT", qc_raw)

    try:
        qc_transpiled = transpile_with_qiskit_compiler(qc_raw, backend=backend)
    except Exception as exc:
        print(f"\n[ERROR] Qiskit transpilation failed for {title}: {type(exc).__name__}: {exc}")
        return

    print_resource_report("TRANSPILED CIRCUIT", qc_transpiled)


def main():
    print("Qiskit QCNN transpiler comparison")
    print("N_QUBITS:", N_QUBITS)
    print("E1_N_QUBITS:", E1_N_QUBITS)
    print("OPTIMIZATION_LEVEL:", OPTIMIZATION_LEVEL)
    print("FAKE_BACKEND_NAMES:", FAKE_BACKEND_NAMES)
    print("USE_FINAL_CLASSIFIER_FOR_HUR:", USE_FINAL_CLASSIFIER_FOR_HUR)

    try:
        backends = get_fake_backends()
    except Exception as exc:
        print(f"[FATAL] {type(exc).__name__}: {exc}")
        return

    ansatzes = OrderedDict({
        "hur6": {
            "conv_fn": hur_convolution_circuit6,
            "conv_params": 6,
            "pool_params": 2,
        },
        "hur8": {
            "conv_fn": hur_convolution_circuit8,
            "conv_params": 10,
            "pool_params": 2,
        },
        "hur9": {
            "conv_fn": hur_convolution_circuit9,
            "conv_params": 15,
            "pool_params": 2,
        },
        "custom_ansatz": {
            "conv_fn": custom_cartan_convolution_block,
            "conv_params": 11,
            "pool_params": 3,
        },
    })

    encodings = ["e3", "e1", "custom"]

    for backend in backends:
        print_backend_info(backend)

        for ansatz_name, ansatz_cfg in ansatzes.items():
            for encoding_name in encodings:
                run_one_case(ansatz_name, ansatz_cfg, encoding_name, backend=backend)


if __name__ == "__main__":
    main()


Qiskit QCNN transpiler comparison
N_QUBITS: 8
E1_N_QUBITS: 9
OPTIMIZATION_LEVEL: 3
FAKE_BACKEND_NAMES: ['FakeSherbrooke', 'FakeTorino']
USE_FINAL_CLASSIFIER_FOR_HUR: False
BACKEND INFO
backend: <qiskit_ibm_runtime.fake_provider.backends.sherbrooke.fake_sherbrooke.FakeSherbrooke object at 0x0000022163646010>
backend name: fake_sherbrooke
basis_gates: ['ecr', 'id', 'rz', 'sx', 'x']
operation_names / supported instructions: ['rz', 'for_loop', 'reset', 'id', 'switch_case', 'ecr', 'x', 'measure', 'if_else', 'delay', 'sx']
num_qubits: 127
coupling_map: [[1, 0], [1, 2], [3, 2], [4, 3], [4, 15], [5, 4], [6, 5], [7, 6], [7, 8], [8, 9], [10, 9], [10, 11], [11, 12], [12, 13], [14, 0], [14, 18], [16, 8], [17, 12], [17, 30], [18, 19], [19, 20], [20, 33], [21, 20], [21, 22], [22, 15], [23, 22], [23, 24], [25, 24], [26, 16], [26, 25], [26, 27], [28, 27], [29, 28], [29, 30], [31, 30], [31, 32], [32, 36], [33, 39], [34, 24], [35, 28], [35, 47], [36, 51], [37, 38], [38, 39], [40, 39], [41, 40], [41, 53]